In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [14]:
train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

print('Train columns:', train.columns.tolist())

Train columns: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_status']


In [15]:
grade_mapping = {'A' : 1, 'B' : 2, 'C' : 3, 'D' : 4, 'E' : 5, 'F' : 6, 'G' : 7}
train['loan_grade'] = train['loan_grade'].map(grade_mapping)
test['loan_grade'] = test['loan_grade'].map(grade_mapping)

default_mapping = {'Y' : 1, 'N' : 0}
train['cb_person_default_on_file'] = train['cb_person_default_on_file'].map(default_mapping)
test['cb_person_default_on_file'] = test['cb_person_default_on_file'].map(default_mapping)

train[['cb_person_default_on_file', 'loan_grade']].head()

,cb_person_default_on_file,loan_grade
0,0,2
1,0,3
2,0,1
3,0,2
4,0,1


In [16]:
train = pd.get_dummies(train, columns=['person_home_ownership', 'loan_intent'], dtype=int)
test = pd.get_dummies(test, columns=['person_home_ownership', 'loan_intent'], dtype=int)
train.head()

,person_age,person_income,person_emp_length,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,37,35000,0.0,2,6000,11.49,0.17,0,14,0,0,0,0,1,0,1,0,0,0,0
1,22,56000,6.0,3,4000,13.35,0.07,0,2,0,0,0,1,0,0,0,0,1,0,0
2,29,28800,8.0,1,6000,8.90,0.21,0,10,0,0,0,1,0,0,0,0,0,1,0
3,30,70000,14.0,2,12000,11.11,0.17,0,5,0,0,0,0,1,0,0,0,0,0,1
4,22,60000,2.0,1,6000,6.92,0.10,0,3,0,0,0,0,1,0,0,0,1,0,0


In [17]:
x = train.drop(columns=['loan_status'])
y = train['loan_status']

In [18]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [19]:
num_cols = ['person_age', 'person_income', 'person_emp_length',
            'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']

scaler = StandardScaler()

x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
x_val[num_cols] = scaler.fit_transform(x_val[num_cols])
test[num_cols] = scaler.fit_transform(test[num_cols])

In [21]:
print('x_train:', x_train.shape)
print('x_val:', x_val.shape)
print('y_train:', y_train.shape)
print('y_val:', y_val.shape)


x_train: (46916, 19)
x_val: (11729, 19)
y_train: (46916,)
y_val: (11729,)


## MODELS

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

lr_model = LogisticRegression(random_state=42, class_weight='balanced',max_iter=1000)
lr_model.fit(x_train, y_train)
y_pred_lr = lr_model.predict(x_val)

print('Logistic Regression:')
print(classification_report(y_val,y_pred_lr))
print('ROC-AUC:', roc_auc_score(y_val, lr_model.predict_proba(x_val)[:,1]))

Logistic Regression:
              precision    recall  f1-score   support

           0       0.97      0.81      0.88     10087
           1       0.42      0.83      0.55      1642

    accuracy                           0.81     11729
   macro avg       0.69      0.82      0.72     11729
weighted avg       0.89      0.81      0.84     11729

ROC-AUC: 0.8967060266304346
